In [1]:
from functools import partial
from notebooks._utils import calculate_series_ensemble_accuracy
from notebooks._utils import calculate_parallel_ensemble_accuracy

ds_name = "myriadlama-debug"
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"

def get_layers(model: str):
    if model.startswith("llama3.2_1b"):
        layers = [8, 10, 12, 14, 16]
    elif model.startswith("llama3.2_3b"):
        layers = [16, 19, 22, 25, 28]
    elif model.startswith("llama3.1_8b"):
        layers = [16, 20, 24, 28, 32]
    elif model.startswith("qwen2.5_3b"):
        layers = [20, 24, 28, 32, 36]
    elif model.startswith("qwen2.5_7b"):
        layers = [16, 19, 22, 25, 28]
    elif model.startswith("qwen2.5_14b"):
        layers = [24, 30, 36, 42, 48]
    else:
        raise NotImplementedError(f"Layers not defined for model {model}")
    return layers


In [11]:
# for model_name in ["llama3.2_1b", "llama3.2_3b", "llama3.1_8b", "qwen2.5_3b", "qwen2.5_7b", ]:
# for model_name in ["llama3.1_8b"]:
model_name = "llama3.2_3b"
print(f"\n=================== Model: {model_name} ===================")
dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."

report_acc_base_setting = partial(
    calculate_parallel_ensemble_accuracy, 
    dump_file_prefix=dump_file_prefix, repeat_paras=False,
    num_paraphrases=5, num_fewshots=5, use_generation=True)

print("---- Calculating baseline ----")
df = calculate_series_ensemble_accuracy(
    dump_file_prefix=dump_file_prefix, 
    single_para_qapair=True, explicit_prompts=False, repeat_paras=False, 
    modifyattn=False, modifyrope=False, scale_score=False, 
    num_paraphrases=1, num_fewshots=5)


print("\n---- Logits-based Ensemble (Average) ----")
report_acc_base_setting(logits_ensemble_method="avg")

print("\n---- Logits-based Ensemble (Maximum) ----")
report_acc_base_setting(logits_ensemble_method="max")

single_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="layer_output_avg", multilayer=False)
multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="layer_output_avg", multilayer=True)

single_ffnavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_avg", multilayer=False)
multip_ffnavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_avg", multilayer=True)

single_ffnmax_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_max", multilayer=False)
multip_ffnmax_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_max", multilayer=True)



=================== Model: llama3.2_3b ===================
---- Calculating baseline ----
Acc: 0.4640 ==> 🏷️ 1paras 5shots 1QA      (Baseline)

---- Logits-based Ensemble (Average) ----
Acc: 0.5640 ==> 🏷️ 5paras 5shots  None layerNone  alpha1.0 token-all

---- Logits-based Ensemble (Maximum) ----
Acc: 0.5450 ==> 🏷️ 5paras 5shots  None layerNone  alpha1.0 token-all


In [12]:
for alpha in [0.25, 0.5, 0.75, 1.0]:
    for token_mode in ["last"]:
        for layer in get_layers(model_name):
            # print(f"\n---- Single Layer Avg Ensemble (token_mode={token_mode}, layer={layer}) ----")
            # single_layavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)
            print(f"\n---- Multi Layer Avg Ensemble (token_mode={token_mode}) ----")
            multip_layavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)


---- Multi Layer Avg Ensemble (token_mode=last) ----
Acc: 0.5570 ==> 🏷️ 5paras 5shots  layer_output_avg layer16 Multilayer alpha0.25 token-last

---- Multi Layer Avg Ensemble (token_mode=last) ----
File ./logits.avg.avglayer.layer19.alpha25.token-last.multilayer.5samples.5paras.feather does not exist!

---- Multi Layer Avg Ensemble (token_mode=last) ----
File ./logits.avg.avglayer.layer22.alpha25.token-last.multilayer.5samples.5paras.feather does not exist!

---- Multi Layer Avg Ensemble (token_mode=last) ----
File ./logits.avg.avglayer.layer25.alpha25.token-last.multilayer.5samples.5paras.feather does not exist!

---- Multi Layer Avg Ensemble (token_mode=last) ----
File ./logits.avg.avglayer.layer28.alpha25.token-last.multilayer.5samples.5paras.feather does not exist!

---- Multi Layer Avg Ensemble (token_mode=last) ----
Acc: 0.5530 ==> 🏷️ 5paras 5shots  layer_output_avg layer16 Multilayer alpha0.5 token-last

---- Multi Layer Avg Ensemble (token_mode=last) ----
Acc: 0.5550 ==> 🏷️ 5p

In [13]:
alpha=0.5
for token_mode in ["last"]:
    for layer in get_layers(model_name):
        print(f"\n---- Single FFN Activation Avg Ensemble (token_mode={token_mode}) ----")
        single_ffnavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)
        print(f"\n---- Multi FFN Activation Avg Ensemble (token_mode={token_mode}) ----")
        multip_ffnavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)


---- Single FFN Activation Avg Ensemble (token_mode=last) ----
File ./logits.avg.avgffn.layer16.alpha50.token-last.5samples.5paras.feather does not exist!

---- Multi FFN Activation Avg Ensemble (token_mode=last) ----
Acc: 0.5500 ==> 🏷️ 5paras 5shots  ffn_activation_avg layer16 Multilayer alpha0.5 token-last

---- Single FFN Activation Avg Ensemble (token_mode=last) ----
File ./logits.avg.avgffn.layer19.alpha50.token-last.5samples.5paras.feather does not exist!

---- Multi FFN Activation Avg Ensemble (token_mode=last) ----


Acc: 0.5540 ==> 🏷️ 5paras 5shots  ffn_activation_avg layer19 Multilayer alpha0.5 token-last

---- Single FFN Activation Avg Ensemble (token_mode=last) ----
File ./logits.avg.avgffn.layer22.alpha50.token-last.5samples.5paras.feather does not exist!

---- Multi FFN Activation Avg Ensemble (token_mode=last) ----
Acc: 0.5530 ==> 🏷️ 5paras 5shots  ffn_activation_avg layer22 Multilayer alpha0.5 token-last

---- Single FFN Activation Avg Ensemble (token_mode=last) ----
File ./logits.avg.avgffn.layer25.alpha50.token-last.5samples.5paras.feather does not exist!

---- Multi FFN Activation Avg Ensemble (token_mode=last) ----
Error reading /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/llama3.2_3b/myriadlama.logits.avg.avgffn.layer25.alpha50.token-last.multilayer.5samples.5paras.feather: Verification of flatbuffer-encoded Footer failed.

---- Single FFN Activation Avg Ensemble (token_mode=last) ----
Acc: 0.5660 ==> 🏷️ 5paras 5shots  ffn_activation_avg layer28  alpha0.5 token-las

In [14]:
alpha=0.5
for token_mode in ["last"]:
    for layer in get_layers(model_name):
        print(f"\n---- Single FFN Activation Max Ensemble (token_mode={token_mode}) ----")
        single_ffnmax_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)
        print(f"\n---- Multi FFN Activation Max Ensemble (token_mode={token_mode}) ----")   
        multip_ffnmax_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)
    


---- Single FFN Activation Max Ensemble (token_mode=last) ----
File ./logits.avg.maxffn.layer16.alpha50.token-last.5samples.5paras.feather does not exist!

---- Multi FFN Activation Max Ensemble (token_mode=last) ----
File ./logits.avg.maxffn.layer16.alpha50.token-last.multilayer.5samples.5paras.feather does not exist!

---- Single FFN Activation Max Ensemble (token_mode=last) ----
File ./logits.avg.maxffn.layer19.alpha50.token-last.5samples.5paras.feather does not exist!

---- Multi FFN Activation Max Ensemble (token_mode=last) ----
File ./logits.avg.maxffn.layer19.alpha50.token-last.multilayer.5samples.5paras.feather does not exist!

---- Single FFN Activation Max Ensemble (token_mode=last) ----
File ./logits.avg.maxffn.layer22.alpha50.token-last.5samples.5paras.feather does not exist!

---- Multi FFN Activation Max Ensemble (token_mode=last) ----
File ./logits.avg.maxffn.layer22.alpha50.token-last.multilayer.5samples.5paras.feather does not exist!

---- Single FFN Activation Max En